# 02 - Demonstracija generisanja multimodalnih embeddinga

## Vision-Language Clothing Retrieval

Ova Jupyter sveska demonstrira pipeline za generisanje multimodalnih embeddinga implementiran u okviru projekta.

Pipeline koristi:

- ResNet10 za generisanje image embeddinga,
- DistilBERT za generisanje text embeddinga,
- PyTorch Dataset i DataLoader za batch obradu.

Za demonstraciju se koristi mali broj uzoraka kako bi se prikazao tok obrade bez ponovnog generisanja embeddinga za kompletan skup podataka.

Na kraju sveske proveravaju se i već generisani embedding fajlovi za train, validation i test skup.

In [1]:
from pathlib import Path

import torch

from vision_language_clothing_retrieval.dataset.deepfashion import (
    DeepFashionDatasetLoader,
)


In [2]:
DATA_DIR = Path("../data")
IMAGES_DIR = DATA_DIR / "images"
CAPTIONS_PATH = DATA_DIR / "captions.json"

EMBEDDINGS_DIR = Path("../embeddings")


In [3]:
loader = DeepFashionDatasetLoader(
    str(IMAGES_DIR),
    str(CAPTIONS_PATH),
)

samples = list(loader.load())

demo_samples = samples[:8]

print(f"Ukupan broj uzoraka: {len(samples):,}")
print(f"Broj uzoraka za demonstraciju: {len(demo_samples)}")


Ukupan broj uzoraka: 42,544
Broj uzoraka za demonstraciju: 8


## 1. Generisanje embeddinga

U ovoj demonstraciji koristi se mali broj uzoraka kako bi se prikazao kompletan tok generisanja embeddinga.

Za svaku sliku generiše se image embedding dimenzije 512, dok se za odgovarajući tekst generiše text embedding dimenzije 768.

Kompletan skup embeddinga prethodno je generisan glavnim pipeline-om. Ova sveska demonstrira proces generisanja na manjem broju uzoraka i zatim proverava već generisane embedding fajlove za train, validation i test skup.

## 2. Inicijalizacija embedding modela

Za generisanje embeddinga koriste se dve pretrained komponente:

- **ResNet10** za vizuelnu reprezentaciju slike,
- **DistilBERT** za tekstualnu reprezentaciju captiona.

Image embedding ima 512 dimenzija, dok text embedding ima 768 dimenzija.

Modeli se koriste u inference režimu, bez računanja gradijenata.

In [4]:
from vision_language_clothing_retrieval.embeddings.image_encoder import (
    ResNet10ImageEncoder,
)
from vision_language_clothing_retrieval.embeddings.text_encoder import (
    DistilBERTTextEncoder,
)
from transformers import logging

logging.set_verbosity_error()
image_encoder = ResNet10ImageEncoder()
text_encoder = DistilBERTTextEncoder()

print("Image encoder: ResNet10")
print("Text encoder: DistilBERT")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

Image encoder: ResNet10
Text encoder: DistilBERT


In [5]:
sample = demo_samples[0]

image_embedding = image_encoder.encode(sample.image_path)
text_embedding = text_encoder.encode(sample.text)

print("Primer embeddinga")
print("-" * 50)
print(f"Sample ID: {sample.sample_id}")
print(f"Image embedding dimenzija: {len(image_embedding)}")
print(f"Text embedding dimenzija:  {len(text_embedding)}")


Primer embeddinga
--------------------------------------------------
Sample ID: MEN-Denim-id_00000080-01_7_additional
Image embedding dimenzija: 512
Text embedding dimenzija:  768


In [6]:
print("\nPrvih 10 vrednosti image embeddinga:")
print(image_embedding[:10])

print("\nPrvih 10 vrednosti text embeddinga:")
print(text_embedding[:10])



Prvih 10 vrednosti image embeddinga:
[0.02872047945857048, 0.1405799388885498, 0.24384735524654388, 0.0, 0.07093493640422821, 0.014471682719886303, 0.0, 0.11629139631986618, 0.0321081206202507, 0.11945213377475739]

Prvih 10 vrednosti text embeddinga:
[-0.11893105506896973, -0.44596797227859497, -0.1598045378923416, -0.12499365210533142, -0.08145154267549515, 0.1876610517501831, 0.21219401061534882, 0.5290815234184265, -0.3336390256881714, -0.384691447019577]


## 3. Batch generisanje embeddinga

Za batch obradu koristi se `TorchClothingDataset` zajedno sa `MultimodalCollator` klasom.

`MultimodalCollator` priprema slike i tekstualne opise u formatu koji očekuje `EmbeddingGenerator`.

Za demonstraciju se koristi 8 uzoraka.

In [7]:
from torch.utils.data import DataLoader

from vision_language_clothing_retrieval.dataset.torch_adapter import (
    TorchClothingDataset,
    MultimodalCollator,
)

demo_dataset = TorchClothingDataset(demo_samples)

demo_dataloader = DataLoader(
    demo_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=MultimodalCollator(),
)

batch = next(iter(demo_dataloader))

print("Oblik image batch-a:", batch["images"].shape)
print("Oblik input_ids:", batch["input_ids"].shape)
print("Oblik attention_mask:", batch["attention_mask"].shape)
print("Broj sample ID-jeva:", len(batch["sample_ids"]))


Oblik image batch-a: torch.Size([8, 3, 224, 224])
Oblik input_ids: torch.Size([8, 59])
Oblik attention_mask: torch.Size([8, 59])
Broj sample ID-jeva: 8


## 4. Generisanje embeddinga

Pripremljeni batch prosleđuje se `EmbeddingGenerator` komponenti.

Generator koristi ResNet10 za slike i DistilBERT za tekst. Generisanje se izvršava u inference režimu, bez računanja gradijenata.

In [8]:
from vision_language_clothing_retrieval.embeddings.generator import (
    EmbeddingGenerator,
)

generator = EmbeddingGenerator(
    image_model=image_encoder.model,
    text_model=text_encoder.model,
    device="cpu",
)

result = generator.generate(demo_dataloader)
print("Broj generisanih uzoraka:", len(result["sample_ids"]))
print("Image embeddings:", result["image_embeddings"].shape)
print("Text embeddings:", result["text_embeddings"].shape)

Processed 1/1 batches
Broj generisanih uzoraka: 8
Image embeddings: torch.Size([8, 512])
Text embeddings: torch.Size([8, 768])


In [9]:
assert len(result["sample_ids"]) == 8
assert result["image_embeddings"].shape == (8, 512)
assert result["text_embeddings"].shape == (8, 768)

print("Embedding generation demo: PASS")


Embedding generation demo: PASS


## 5. Provera konačnih embedding fajlova

Nakon završetka embedding pipeline-a, checkpoint fajlovi su spojeni u tri konačna PyTorch fajla:

- `train.pt`
- `validation.pt`
- `test.pt`

U nastavku se proveravaju njihova veličina i dimenzionalnost.

In [10]:
embedding_files = {
    "Train": EMBEDDINGS_DIR / "train.pt",
    "Validation": EMBEDDINGS_DIR / "validation.pt",
    "Test": EMBEDDINGS_DIR / "test.pt",
}

for split_name, path in embedding_files.items():
    data = torch.load(path, weights_only=True)

    print(f"{split_name}:")
    print(f"  Samples: {len(data['sample_ids']):,}")
    print(f"  Image embeddings: {data['image_embeddings'].shape}")
    print(f"  Text embeddings:  {data['text_embeddings'].shape}")
    print()


Train:
  Samples: 34,047
  Image embeddings: torch.Size([34047, 512])
  Text embeddings:  torch.Size([34047, 768])

Validation:
  Samples: 4,354
  Image embeddings: torch.Size([4354, 512])
  Text embeddings:  torch.Size([4354, 768])

Test:
  Samples: 4,136
  Image embeddings: torch.Size([4136, 512])
  Text embeddings:  torch.Size([4136, 768])



In [11]:
expected = {
    "Train": 34047,
    "Validation": 4354,
    "Test": 4136,
}

for split_name, path in embedding_files.items():
    data = torch.load(path, weights_only=True)

    assert len(data["sample_ids"]) == expected[split_name]
    assert data["image_embeddings"].shape == (
        expected[split_name],
        512,
    )
    assert data["text_embeddings"].shape == (
        expected[split_name],
        768,
    )

print("Final embedding verification: PASS")


Final embedding verification: PASS


## Zaključak

Demonstriran je kompletan tok generisanja multimodalnih embeddinga:

1. učitavanje image-caption uzoraka,
2. priprema batch-a pomoću PyTorch `DataLoader`-a,
3. preprocesiranje slika i tekstualnih opisa,
4. generisanje image embeddinga pomoću ResNet10,
5. generisanje text embeddinga pomoću DistilBERT-a,
6. čuvanje embeddinga u PyTorch formatu,
7. provera konačnih embedding fajlova.

Za svaki uzorak generiše se:

- image embedding dimenzije **512**,
- text embedding dimenzije **768**.

Konačni embedding fajlovi sadrže kompletne reprezentacije train, validation i test skupova i predstavljaju izlaz ove komponente za downstream retrieval sistem.
